# Desafío por equipos · De modelo experimental a producción

**Caso:** NovaRetail · **Siete equipos, siete desafíos distintos.** Cada equipo resuelve el suyo.

Un modelo que predice el abandono de clientes vive en un notebook. Cada equipo diseña **la ruta notebook → paquete → API → registro → dashboard** para su caso y la prueba contra una simulación. Cada equipo entrega:

1. **Un diagrama de despliegue** de la ruta, con el control de cada etapa
2. **El contrato revisado**, con sus cambios
3. **Cinco pruebas**
4. **Un tablero mínimo** de monitoreo
5. **La decisión de rollback**: disparador, responsable y pasos

Incluyan **versionado, prueba de humo, rollback, latencia, drift y un responsable humano**. Cada decisión lleva **una alternativa que se descartó**. Si preguntan «¿está bien?», el facilitador va a devolver cinco preguntas, en orden: *¿qué decisión habilita? ¿cuál es la población y el período? ¿cómo evitaron fuga o sesgo? ¿qué validaron? ¿qué no pueden afirmar?*

Los datos son ficticios y se generan desde el código. No hay credenciales ni infraestructura real.

## Paso 0 · Instalación

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "yaml": "pyyaml", "joblib": "joblib", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## Preparación

Esta celda carga las librerías, genera NovaRetail, entrena las versiones 1 y 2 del modelo y define el servicio. **No hace falta leerla**: correla una sola vez. Trae también los laboratorios de los siete casos.

In [ ]:
# ═══════════════════════════ KIT DE PRODUCCIÓN · NovaRetail (datos ficticios, generados por código)
# De un notebook a un servicio: contrato, validación, versionado, registro de predicciones y monitoreo.
import hashlib, json, os, time, warnings
import numpy as np, pandas as pd, yaml, joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


def datos_limpios():
    df = generar_novaretail().drop_duplicates("id_cliente").reset_index(drop=True)
    df.loc[(df.edad < 18) | (df.edad > 100), "edad"] = np.nan
    df["satisfaccion_faltante"] = df.satisfaccion.isna().astype(int); return df
DF = datos_limpios()

NUM_SERV = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion"]
CAT_SERV = ["canal", "segmento", "region"]
NUM_MODELO = NUM_SERV + ["satisfaccion_faltante"]; OBJETIVO = "abandono_90d"

def armar_pipeline(modelo, num=NUM_MODELO, cat=CAT_SERV):
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
                             ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("pre", pre), ("modelo", modelo)])
def partir(df, columnas):
    return train_test_split(df[columnas], df[OBJETIVO], test_size=0.25, stratify=df[OBJETIVO], random_state=42)
_XTR, _XTE, _YTR, _YTE = partir(DF, NUM_MODELO + CAT_SERV)
MODELO_V1 = armar_pipeline(LogisticRegression(max_iter=3000)).fit(_XTR, _YTR)
MODELO_V2 = armar_pipeline(HistGradientBoostingClassifier(max_depth=3, random_state=7)).fit(_XTR, _YTR)
AUC_V1 = float(roc_auc_score(_YTE, MODELO_V1.predict_proba(_XTE)[:, 1])); AUC_V2 = float(roc_auc_score(_YTE, MODELO_V2.predict_proba(_XTE)[:, 1]))

# ═══════ EL CONTRATO: un archivo OpenAPI que describe qué entra y qué sale
CONTRATO_BORRADOR = """openapi: 3.0.3
info:
  title: Servicio de riesgo de abandono NovaRetail
  version: "0.1"
paths:
  /predict:
    post:
      summary: Probabilidad de abandono en 90 días
      requestBody:
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/PrediccionRequest"
      responses:
        "200":
          description: Predicción
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/PrediccionResponse"
        "500":
          description: Error
components:
  schemas:
    PrediccionRequest:
      type: object
      required: [id_cliente]
      properties:
        id_cliente: {type: string}
        edad: {type: number}
        antiguedad_meses: {type: integer}
        compras_12m: {type: integer}
        ticket_promedio: {type: number}
        dias_desde_ultima_compra: {type: integer}
        contactos_soporte_90d: {type: integer}
        usa_app: {type: integer}
        descuento_promedio: {type: number}
        satisfaccion: {type: number}
        canal: {type: string}
        segmento: {type: string}
        region: {type: string}
    PrediccionResponse:
      type: object
      properties:
        probabilidad: {type: number}
"""
def leer_contrato(texto_o_ruta):
    texto = open(texto_o_ruta, encoding="utf-8").read() if os.path.exists(str(texto_o_ruta)) else texto_o_ruta
    return yaml.safe_load(texto)
def reglas_desde_contrato(contrato):
    esquema = contrato["components"]["schemas"]["PrediccionRequest"]
    return {"requeridos": list(esquema.get("required", [])), "propiedades": esquema.get("properties", {}), "extras_permitidos": esquema.get("additionalProperties", True) is not False}
def validar(payload, reglas):
    """Devuelve la lista de errores {campo, motivo}. Vacía = el payload cumple el contrato."""
    err = []; props = reglas["propiedades"]
    for c in reglas["requeridos"]:
        if c not in payload: err.append({"campo": c, "motivo": "falta"})
    for c, v in payload.items():
        if c not in props:
            if not reglas["extras_permitidos"]: err.append({"campo": c, "motivo": "campo no permitido"})
            continue
        p = props[c]
        if v is None:
            if not p.get("nullable"): err.append({"campo": c, "motivo": "no admite nulo"})
            continue
        t = p.get("type")
        if t == "integer" and (isinstance(v, bool) or not isinstance(v, int)): err.append({"campo": c, "motivo": "tiene que ser entero"}); continue
        if t == "number" and (isinstance(v, bool) or not isinstance(v, (int, float))): err.append({"campo": c, "motivo": "tiene que ser número"}); continue
        if t == "string" and not isinstance(v, str): err.append({"campo": c, "motivo": "tiene que ser texto"}); continue
        if "minimum" in p and v < p["minimum"]: err.append({"campo": c, "motivo": f"menor que {p['minimum']}"})
        if "maximum" in p and v > p["maximum"]: err.append({"campo": c, "motivo": f"mayor que {p['maximum']}"})
        if "enum" in p and v not in p["enum"]: err.append({"campo": c, "motivo": f"valor no permitido (admite {p['enum']})"})
    return err
# ═══════ EL SERVICIO: carga del artefacto, esquema, transformación, inferencia, respuesta y registro
def transformar(payload):
    fila = {c: pd.to_numeric(payload.get(c), errors="coerce") for c in NUM_SERV}
    fila["satisfaccion_faltante"] = int(pd.isna(fila["satisfaccion"])); fila.update({c: (payload.get(c) if payload.get(c) is not None else np.nan) for c in CAT_SERV})
    return pd.DataFrame([fila])
def hash_entrada(payload): return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:10]
class Servicio:
    """Simula un POST /v1/predict: valida contra el contrato, puntúa con una versión del modelo y deja un registro."""
    def __init__(self, contrato, modelo=None, version="v1"):
        self.contrato = contrato if isinstance(contrato, dict) else leer_contrato(contrato); self.reglas = reglas_desde_contrato(self.contrato)
        self.modelo, self.version, self.registro, self.relojes = modelo or MODELO_V1, version, [], 0
    def responder(self, payload):
        t0 = time.perf_counter(); errores = validar(payload, self.reglas)
        if errores: resp = {"status": 422, "errores": errores}
        else:
            try: resp = {"status": 200, "probabilidad": round(float(self.modelo.predict_proba(transformar(payload))[0, 1]), 4)}
            except Exception as e: resp = {"status": 500, "detalle": type(e).__name__}
        resp["version_modelo"] = self.version; resp["version_contrato"] = str(self.contrato["info"]["version"])
        self.relojes += 1; self.registro.append({"n": self.relojes, "version_modelo": self.version, "status": resp["status"], "entrada": hash_entrada(payload), "probabilidad": resp.get("probabilidad"), "latencia_ms": (time.perf_counter() - t0) * 1000})
        return resp
def cliente_ejemplo(i=0, **cambios):
    f = DF.iloc[i]; p = {c: (None if pd.isna(f[c]) else (int(f[c]) if c in ("antiguedad_meses", "compras_12m", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app") else float(f[c]))) for c in NUM_SERV}
    p.update({c: f[c] for c in CAT_SERV}); p["id_cliente"] = f["id_cliente"]; p.update(cambios); return p
def _tipico():
    p = cliente_ejemplo(0); p.update({"edad": 41.0, "antiguedad_meses": 30, "compras_12m": 7, "ticket_promedio": 48.5, "dias_desde_ultima_compra": 55, "contactos_soporte_90d": 1, "usa_app": 1, "descuento_promedio": 0.1, "satisfaccion": 4.0, "canal": "web", "segmento": "Plus", "region": "Centro"}); return p
CASOS_CONTRATO = [("típico, todo válido", _tipico(), False), ("satisfacción nula (permitida)", {**_tipico(), "satisfaccion": None}, False),
    ("falta un campo que usa el modelo", {k: v for k, v in _tipico().items() if k != "dias_desde_ultima_compra"}, True), ("texto en un campo numérico", {**_tipico(), "compras_12m": "muchas"}, True),
    ("edad imposible (150)", {**_tipico(), "edad": 150.0}, True), ("edad imposible (−1)", {**_tipico(), "edad": -1.0}, True), ("días negativos", {**_tipico(), "dias_desde_ultima_compra": -5}, True),
    ("segmento que no existe", {**_tipico(), "segmento": "Gold"}, True), ("canal nuevo sin avisar", {**_tipico(), "canal": "marketplace"}, True),
    ("campo con fuga (motivo_baja)", {**_tipico(), "motivo_baja": "precio"}, True), ("campo extra inesperado", {**_tipico(), "cupon": "X10"}, True),
    ("descuento fuera de rango", {**_tipico(), "descuento_promedio": 1.7}, True), ("usa_app fuera de {0, 1}", {**_tipico(), "usa_app": 2}, True), ("payload vacío", {}, True)]
def probar_reglas(reglas, mostrar=True):
    filas = []
    for nombre, payload, debe_rechazar in CASOS_CONTRATO:
        rechaza = bool(validar(payload, reglas)); ok = rechaza == debe_rechazar
        filas.append({"caso": nombre, "debería": "rechazar" if debe_rechazar else "aceptar", "el contrato": "rechaza" if rechaza else "acepta", "ok": "✔" if ok else "✘"})
    t = pd.DataFrame(filas)
    if mostrar: print(t.to_string(index=False))
    return t

# ═══════ EL REGISTRO DE VERSIONES: artefacto + ficha
def registrar_version(carpeta, version, modelo, auc, algoritmo, semilla=None):
    ruta = os.path.join(carpeta, version); os.makedirs(ruta, exist_ok=True); joblib.dump(modelo, os.path.join(ruta, "modelo.joblib"))
    ficha = {"version": version, "algoritmo": algoritmo, "auc_prueba": round(auc, 4), "columnas_entrada": NUM_MODELO + CAT_SERV, "version_contrato": "1.0.0", "semilla": semilla,
             "hash_datos_entrenamiento": hashlib.sha256(pd.util.hash_pandas_object(DF[NUM_MODELO + CAT_SERV + [OBJETIVO]], index=False).values.tobytes()).hexdigest()[:12], "filas_entrenamiento": int(len(_XTR))}
    json.dump(ficha, open(os.path.join(ruta, "ficha.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2); return ficha
def cargar_version(carpeta, version): return joblib.load(os.path.join(carpeta, version, "modelo.joblib")), json.load(open(os.path.join(carpeta, version, "ficha.json"), encoding="utf-8"))

def escribir_contratos(carpeta=".", revisado=False):
    """Deja el contrato borrador (y, si se pide, el revisado) como archivos .yaml en la carpeta."""
    os.makedirs(carpeta, exist_ok=True); open(os.path.join(carpeta, "contrato-api.yaml"), "w", encoding="utf-8").write(CONTRATO_BORRADOR)
    if revisado: open(os.path.join(carpeta, "contrato-api.revisado.yaml"), "w", encoding="utf-8").write(CONTRATO_REVISADO)

# ═══════ CASOS CONOCIDOS PARA LA PRUEBA DE HUMO
def _golden(clase):
    base = _tipico(); tip = [{**base, "dias_desde_ultima_compra": d, "segmento": s, "contactos_soporte_90d": c} for d, s, c in ((40, "Plus", 1), (50, "Esencial", 2), (55, "Premium", 0), (45, "Plus", 3), (60, "Esencial", 1))]
    bor = [{**base, "dias_desde_ultima_compra": d, "segmento": s, "contactos_soporte_90d": c} for d, s, c in ((90, "Esencial", 3), (120, "Plus", 2), (150, "Esencial", 4), (180, "Premium", 1), (240, "Esencial", 5), (300, "Plus", 3))]
    return {"tipicos": tip, "borde": bor, "mezcla": tip[:3] + bor[:3]}[clase]
def _probas(modelo, casos, defecto=False):
    return np.array([float(modelo.predict_proba(transformar({**c, "dias_desde_ultima_compra": min(c["dias_desde_ultima_compra"], 60)} if defecto else c))[0, 1]) for c in casos])



# ═══════════════════════════ LOS SIETE LABORATORIOS (cada uno prueba una decisión del equipo y devuelve preguntas)
ENTEROS = ("antiguedad_meses", "compras_12m", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app")
def _reglas_desde_parametros(p):
    props = {c: {"type": "string"} for c in ["id_cliente"] + CAT_SERV}; props.update({c: {"type": "integer" if c in ENTEROS else "number"} for c in NUM_SERV})
    for c, (lo, hi) in (p.get("rangos") or {}).items(): props[c]["minimum"], props[c]["maximum"] = lo, hi
    for c, vals in (p.get("valores_permitidos") or {}).items(): props[c]["enum"] = list(vals)
    for c in p.get("admite_nulo") or []: props[c]["nullable"] = True
    return {"requeridos": list(p.get("campos_obligatorios") or []), "propiedades": props, "extras_permitidos": not p.get("rechazar_campos_extra", False)}
_PREG_CONTRATO = {"falta un campo que usa el modelo": "Si llega un cliente sin un dato que usa el modelo, ¿qué pasa? ¿Lo reemplaza en silencio y nadie se entera?", "edad imposible (150)": "¿Qué rango de edad tiene sentido? ¿Qué pasa con una edad de 150?", "edad imposible (−1)": "¿Qué rango de edad tiene sentido? ¿Qué pasa con una edad negativa?",
    "días negativos": "¿Qué rango tienen los días desde la última compra?", "segmento que no existe": "¿Qué valores admite el segmento? ¿Qué hace el modelo con uno que nunca vio?", "canal nuevo sin avisar": "¿Qué valores admite el canal? Si aparece uno nuevo, ¿preferís que el servicio lo puntúe o que avise?",
    "campo con fuga (motivo_baja)": "¿Qué hace el servicio con un campo que no conoce? ¿Y si ese campo es una variable que solo existe después de la baja?", "campo extra inesperado": "¿Qué hace el servicio con un campo que no figura en el contrato?", "descuento fuera de rango": "¿Qué rango tiene el descuento promedio?",
    "usa_app fuera de {0, 1}": "¿Qué valores admite usa_app?", "payload vacío": "¿Qué campos son obligatorios? Con un pedido vacío, ¿el servicio responde algo?", "satisfacción nula (permitida)": "¿Qué campos pueden venir nulos? Hay un caso válido que tu contrato está rechazando."}
def simular_contrato(p):
    t = probar_reglas(_reglas_desde_parametros(p), mostrar=True); mal = t[t.ok == "✘"]; print(f"\nCasos mal resueltos: {len(mal)} de {len(t)}")
    pistas = [_PREG_CONTRATO[c] for c in mal.caso if c in _PREG_CONTRATO]; return {"ok": len(mal) == 0, "pistas": pistas}

CAT_POR_MODELO = ["version", "algoritmo", "auc_prueba", "columnas_entrada", "version_contrato", "semilla", "hash_datos_entrenamiento", "filas_entrenamiento", "fecha_entrenamiento", "aprobado_por", "commit_codigo", "entorno"]
CAT_POR_PREDICCION = ["timestamp", "request_id", "version_modelo", "version_contrato", "hash_entrada", "entrada_guardada", "probabilidad", "status", "latencia_ms"]
def simular_registro(p):
    m, q = set(p.get("por_modelo") or []), set(p.get("por_prediccion") or []); pistas = []
    preguntas = [("Un cliente reclama una decisión de hace 20 días: ¿qué versión del modelo respondió?", ({"timestamp", "version_modelo"}, {"request_id", "hash_entrada"}), "q"),
                 ("¿Con qué datos se entrenó esa versión?", ({"hash_datos_entrenamiento", "fecha_entrenamiento"},), "m"), ("Reproducí la predicción de ayer.", ({"entrada_guardada", "version_modelo", "version_contrato"},), "q"),
                 ("¿La versión nueva es mejor que la anterior? ¿Con qué métrica?", ({"auc_prueba", "version"},), "m"), ("¿Quién aprobó que esa versión pasara a producción y cuándo?", ({"aprobado_por", "fecha_entrenamiento"},), "m"),
                 ("¿Cuánto tarda el servicio y cuántos pedidos fallaron ayer?", ({"latencia_ms", "status", "timestamp"},), "q")]
    for texto, alternativas, donde in preguntas:
        tiene = q if donde == "q" else m; falta = [sorted(a - tiene) for a in alternativas]; ok = any(not f for f in falta)
        if texto.startswith("Un cliente"): ok = {"timestamp", "version_modelo"} <= q and ({"request_id"} <= q or {"hash_entrada"} <= q)
        print(f"  {'✔' if ok else '✘'} {texto}" + ("" if ok else f"  → falta registrar: {min(falta, key=len)}"))
        if not ok: pistas.append(f"«{texto}» ¿qué tendrías que haber guardado para poder responderla?")
    if "entrada_guardada" in q and (not p.get("retencion_dias") or not (p.get("acceso") or "").strip()): pistas.append("Guardás la entrada completa, que trae el id del cliente: ¿por cuánto tiempo y quién puede leerla?")
    if len(q) > 8 and len(m) > 10: pistas.append("Guardás casi todo: ¿qué campos podrías sacar sin perder ninguna de las preguntas?")
    return {"ok": not pistas, "pistas": pistas}

def simular_humo(p):
    casos = _golden(p.get("casos", "tipicos")); tol = float(p.get("tolerancia", 0.0)); esperado = _probas(MODELO_V1, casos)
    sano, roto = _probas(MODELO_V2, casos), _probas(MODELO_V2, casos, defecto=True); d_sano, d_roto = float(np.max(np.abs(sano - esperado))), float(np.max(np.abs(roto - esperado)))
    pasa_sano, atrapa = d_sano <= tol, d_roto > tol; accion = p.get("accion_si_falla") or ""; resp = (p.get("responsable") or "").strip(); pistas = []; ok = True
    print(f"Casos de la prueba de humo: {p.get('casos', 'tipicos')} ({len(casos)}) · tolerancia {tol:.2f} frente a las respuestas de la versión en producción")
    print(f"  Versión nueva SANA:  diferencia máxima {d_sano:.3f} → {'pasa' if pasa_sano else 'la prueba la BLOQUEA (falsa alarma)'}")
    print(f"  Versión nueva ROTA (el servicio recorta los días a 60):  diferencia máxima {d_roto:.3f} → {'la prueba la ATRAPA' if atrapa else 'la prueba la deja pasar'}")
    if not atrapa: ok = False; pistas.append("La versión rota pasó la prueba de humo: ¿qué clientes tendrían que estar en la prueba para que un error así se note?")
    if not pasa_sano: ok = False; pistas.append("La prueba bloquea una versión sana: ¿qué diferencia es normal entre dos modelos distintos y cuál es un error?")
    minutos = {"rollback_automatico": 1, "alerta": 15 if resp else 120}.get(accion, 240)
    print(f"  Si el defecto llega a producción: se vuelve a la versión anterior en {minutos} minutos" + (" (nadie lo mira hasta que reclaman)" if not accion else ""))
    if not accion: ok = False; pistas.append("Si la prueba falla, ¿qué pasa? ¿Se vuelve atrás solo o alguien decide?")
    elif accion == "alerta" and not resp: ok = False; pistas.append("La alerta suena, pero ¿quién decide volver atrás?")
    return {"ok": ok, "pistas": pistas}

LAT_FIJA, LAT_FILA, N_CRM, N_NOCHE, TOPE_NOCHE_S = 25.0, 0.4, 40, 8000, 300
def simular_latencia(p):
    ic, ma = p.get("interactivo") or {}, p.get("masivo") or {}; obj = float(p.get("objetivo_p95_ms", 300)); pistas = []; ok = True; rng = np.random.default_rng(3); llamada = lambda n: (LAT_FIJA + LAT_FILA * n) * rng.lognormal(0, 0.25, 4000)
    modo = ic.get("modo", "individual"); lote = int(ic.get("tamano_lote", 1)) if modo == "lote" else 1; base, tope = int(ic.get("instancias_base", 1)), int(ic.get("autoescalado_hasta", ic.get("instancias_base", 1)))
    espera = (lote - 1) / N_CRM * 1000 / 2 if modo == "lote" else 0.0; p95 = float(np.percentile(llamada(lote), 95)) + espera; cap = lambda k: k * 1000 / (LAT_FIJA + LAT_FILA * lote) * lote
    print("A · el CRM pide una predicción por ticket (40 por segundo)")
    a_ok = p95 <= obj and cap(base) >= 1.25 * N_CRM; print(f"   p95 estimado {p95:.0f} ms (objetivo {obj:.0f} ms) · capacidad {cap(base):.0f} pedidos por segundo con {base} instancia(s) → {'✔' if a_ok else '✘'}")
    if p95 > obj: ok = False; pistas.append("El p95 supera el objetivo del CRM: ¿qué parte del tiempo es esperar a juntar un lote?")
    if cap(base) < 1.25 * N_CRM: ok = False; pistas.append("Con esas instancias, la capacidad apenas alcanza el volumen de todos los días: ¿qué pasa en una tarde con más tickets?")
    print("B · un proceso nocturno puntúa 8.000 clientes y tiene que terminar en 5 minutos")
    lm, to = int(ma.get("tamano_lote", 1)), float(ma.get("timeout_ms", 1000)); dur = llamada(lm); p99 = float(np.percentile(dur, 99)); total = np.ceil(N_NOCHE / lm) * float(np.mean(dur)) / 1000; b_ok = p99 < to and total <= TOPE_NOCHE_S
    print(f"   lotes de {lm} · {int(np.ceil(N_NOCHE / lm))} llamadas · duración p99 de una llamada {p99:.0f} ms (timeout {to:.0f} ms) · total {total:.0f} s → {'✔' if b_ok else '✘'}")
    if p99 >= to: ok = False; pistas.append("Algunas llamadas del proceso nocturno superan el timeout: ¿qué relación hay entre el tamaño del lote y el tiempo de una llamada?")
    if total > TOPE_NOCHE_S: ok = False; pistas.append("El proceso nocturno no termina en 5 minutos: ¿cuántas llamadas hace?")
    print("C · un pico de tickets: cinco veces el volumen normal"); c_ok = cap(tope) >= 1.25 * 5 * N_CRM; print(f"   capacidad con autoescalado hasta {tope} instancia(s): {cap(tope):.0f} pedidos por segundo (se necesitan {1.25 * 5 * N_CRM:.0f}) → {'✔' if c_ok else '✘'}")
    if not c_ok: ok = False; pistas.append("Si el volumen se multiplica por cinco, ¿hasta cuánto puede crecer el servicio? ¿Qué pasa con los pedidos que no entran?")
    return {"ok": ok, "pistas": pistas}

def _psi(ref, nuevo, bins=10):
    cortes = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1))); cortes[0], cortes[-1] = -np.inf, np.inf
    a = np.histogram(ref, cortes)[0] / len(ref); b = np.histogram(nuevo, cortes)[0] / len(nuevo); a, b = np.clip(a, 1e-4, None), np.clip(b, 1e-4, None); return float(np.sum((b - a) * np.log(b / a)))
def _calcular_deriva():
    ref_d = DF["dias_desde_ultima_compra"].values; ref_p = MODELO_V1.predict_proba(DF[NUM_MODELO + CAT_SERV])[:, 1]; filas, lotes = [], {}
    for d in range(200):
        r = np.random.default_rng(9000 + d); der = float(np.clip((d - 60) / 20, 0, 1)); x = DF.sample(300, replace=True, random_state=9000 + d).reset_index(drop=True)
        x["dias_desde_ultima_compra"] = np.clip((x["dias_desde_ultima_compra"] * (1 + 0.6 * der)).round(), 1, 365).astype(int); nuevo = r.random(300) < 0.25 * der; ok = x[~nuevo].copy()
        p = MODELO_V1.predict_proba(ok[NUM_MODELO + CAT_SERV])[:, 1]; logit = np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4))) + r.normal(0, 0.3 + 3.0 * der, len(p))
        ok["p"] = p; ok["y"] = (r.random(len(p)) < 1 / (1 + np.exp(-logit))).astype(int); lotes[d] = ok
        filas.append({"dia": d, "psi_dias": _psi(ref_d, x["dias_desde_ultima_compra"].values), "psi_score": _psi(ref_p, p), "pct_422": float(nuevo.mean())})
    t = pd.DataFrame(filas); aucs = []
    for d in range(200):
        corte = d - 90; w = pd.concat([lotes[k] for k in range(max(0, corte - 6), corte + 1)]) if corte >= 6 else None; aucs.append(float(roc_auc_score(w["y"], w["p"])) if w is not None else np.nan)
    t["auc_etiquetas"] = aucs; return t
DERIVA_NR = _calcular_deriva(); DIA_DERIVA_NR = 60
def simular_drift(senales):
    pistas = []; ok = True; filas = []; por = {m.get("senal"): m for m in senales}
    for nombre, m in por.items():
        if nombre not in DERIVA_NR.columns: pistas.append(f"«{nombre}» no es una señal de este laboratorio: usá psi_dias, psi_score, pct_422 o auc_etiquetas."); ok = False; continue
        u = float(m.get("umbral", np.inf)); s = DERIVA_NR[nombre]; al = (s < u) if nombre == "auc_etiquetas" else (s > u); falsas = int(al[:DIA_DERIVA_NR].sum()); post = al[DIA_DERIVA_NR:]; primera = int(post.idxmax()) if post.any() else None
        filas.append({"señal": nombre, "umbral": u, "falsas alarmas antes de la deriva": falsas, "primer aviso (día)": primera if primera is not None else "nunca", "demora desde que empieza": (primera - DIA_DERIVA_NR) if primera is not None else "—"})
        if not m.get("accion") or not m.get("responsable"): ok = False; pistas.append(f"«{nombre}»: cuando suena, ¿qué acción se dispara y quién la ejecuta?")
        if falsas > 3: ok = False; pistas.append(f"«{nombre}» suena {falsas} veces en días normales: ¿qué umbral la haría útil sin que la ignoren?")
    print(f"La deriva empieza el día {DIA_DERIVA_NR}: los clientes dormidos aumentan y aparece un canal nuevo. El abandono real se conoce a los 90 días.\n"); print(pd.DataFrame(filas).to_string(index=False) if filas else "(sin señales)")
    tempranas = [f for f in filas if f["demora desde que empieza"] != "—" and f["demora desde que empieza"] < 30]
    if not tempranas: ok = False; pistas.append("Ninguna señal avisa antes de que lleguen los resultados reales: ¿qué se puede mirar mientras tanto?")
    capas = {"entrada": "pct_422" in por or "psi_dias" in por, "datos": bool({"psi_dias", "psi_score"} & set(por)), "modelo": "auc_etiquetas" in por}
    if not capas["modelo"]: pistas.append("¿Cómo confirmás, cuando llegan los resultados reales, que el modelo empeoró?"); ok = False
    if "pct_422" not in por: pistas.append("Si aparece un valor nuevo, el contrato lo rechaza: ¿qué señal te muestra cuántos pedidos se están perdiendo?"); ok = False
    return {"ok": ok and bool(tempranas), "pistas": pistas}

COLUMNAS_FUGA = ["canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion", "satisfaccion_faltante", "motivo_baja"]
def simular_fuga(p):
    cols = list(p.get("columnas") or []); pistas = []; ok = True
    if not cols: return {"ok": False, "pistas": ["Elegí las columnas con las que entrena el modelo."]}
    malas = [c for c in cols if c not in COLUMNAS_FUGA]
    if malas: return {"ok": False, "pistas": [f"{malas} no es una columna de este laboratorio: elegí entre {COLUMNAS_FUGA}"]}
    num = [c for c in cols if c in NUM_MODELO]; cat = [c for c in cols if c in CAT_SERV + ["motivo_baja"]]
    def entrenar(df_tr, df_te, c_num, c_cat):
        m = armar_pipeline(LogisticRegression(max_iter=3000), c_num, c_cat).fit(df_tr[c_num + c_cat], df_tr[OBJETIVO]); return m, float(roc_auc_score(df_te[OBJETIVO], m.predict_proba(df_te[c_num + c_cat])[:, 1]))
    base = DF.copy(); base["motivo_baja"] = base["motivo_baja"].fillna("sin_motivo")      # quien no abandonó no tiene motivo de baja
    tr, te = train_test_split(base, test_size=0.25, stratify=base[OBJETIVO], random_state=42); m, auc = entrenar(tr, te, num, cat)
    prod = te.copy(); prod["motivo_baja"] = "sin_motivo"                  # al puntuar, el cliente todavía no se fue: no tiene motivo
    auc_prod = float(roc_auc_score(prod[OBJETIVO], m.predict_proba(prod[num + cat])[:, 1])); ordenado = base.sort_values("fecha_alta"); corte = int(len(ordenado) * 0.75); _, auc_t = entrenar(ordenado.iloc[:corte], ordenado.iloc[corte:], num, cat)
    print(f"Columnas del modelo: {cols}"); print(f"AUC en la prueba (partición aleatoria): {auc:.3f} · validando con los clientes que se dieron de alta después: {auc_t:.3f} · AUC en producción, cuando motivo_baja no existe todavía: {auc_prod:.3f}")
    if "motivo_baja" in cols: ok = False; pistas.append("Tu modelo parece excelente en la prueba y se cae en producción: ¿qué columna se conoce recién después de que el cliente se va?")
    fuera = [c for c in cols if c not in NUM_SERV + CAT_SERV + ["satisfaccion_faltante"]]
    if fuera and "motivo_baja" not in fuera: ok = False
    if fuera: print(f"Columnas que el contrato no recibe: {fuera}"); pistas.append(f"Entrenás con {fuera}, pero el servicio no las recibe: ¿cómo las vas a tener al momento de puntuar?") if "motivo_baja" not in fuera else None
    if auc - auc_t > 0.03: ok = False; pistas.append("Con clientes más nuevos el AUC baja bastante: ¿tu modelo generaliza o memoriza el período?")
    if len(cols) < 5: ok = False; pistas.append("Con tan pocas variables, ¿qué información de negocio dejaste afuera?")
    return {"ok": ok and not pistas, "pistas": pistas}

CATALOGO_PANELES = {"volumen_total": "Cantidad de pedidos por hora", "volumen_por_region": "Cantidad de pedidos por hora y por región", "tasa_errores_422": "Porcentaje de pedidos rechazados por el contrato", "latencia_p95": "Tiempo de respuesta del 95 % de los pedidos",
    "distribucion_scores": "Cómo se reparten las probabilidades que devuelve el modelo", "psi_variables": "Cuánto cambió cada variable de entrada respecto del entrenamiento", "version_activa": "Qué versión del modelo está respondiendo y desde cuándo", "auc_etiquetas": "AUC con los resultados reales, cuando llegan",
    "uso_de_cpu": "Uso de procesador del servidor", "usuarios_conectados": "Cantidad de personas con el CRM abierto", "filas_en_tabla_de_clientes": "Tamaño de la tabla de clientes"}
INCIDENTES = [("Aparece un canal nuevo y el contrato lo rechaza", {"tasa_errores_422"}), ("Una versión nueva responde el doble de lento", {"latencia_p95"}), ("Alguien activa otra versión del modelo sin avisar", {"version_activa"}),
    ("Aumentan los clientes dormidos: cambia la distribución de los días desde la última compra", {"psi_variables", "distribucion_scores"}), ("El modelo empeora de a poco y los resultados reales tardan 90 días", {"auc_etiquetas"}),
    ("Un corte de integración deja en cero los pedidos de una región", {"volumen_por_region"}), ("Todos los puntajes se pegan a un mismo valor porque llegan valores por defecto", {"distribucion_scores"})]
def simular_tablero(p):
    pan = list(p.get("paneles") or []); pistas = []; ok = True; malos = [x for x in pan if x not in CATALOGO_PANELES]
    if malos: return {"ok": False, "pistas": [f"{malos} no está en el catálogo: {list(CATALOGO_PANELES)}"]}
    print(f"Tablero de {len(pan)} panel(es): {pan}"); cubiertos = 0
    for texto, detectan in INCIDENTES:
        visto = bool(set(pan) & detectan); cubiertos += visto; print(f"  {'✔' if visto else '✘'} {texto}" + ("" if visto else "  → ningún panel lo muestra"))
        if not visto: pistas.append(f"«{texto}»: ¿qué panel lo mostraría?")
    print(f"Incidentes cubiertos: {cubiertos} de {len(INCIDENTES)}")
    if len(pan) > 6: ok = False; pistas.append(f"Tu tablero tiene {len(pan)} paneles: ¿cuál es el mínimo que alcanza para ver todo lo que importa?")
    inutiles = [x for x in pan if x in ("uso_de_cpu", "usuarios_conectados", "filas_en_tabla_de_clientes")]
    if inutiles: ok = False; pistas.append(f"{inutiles}: ¿qué incidente de la lista te avisa?")
    if not (p.get("responsable") or "").strip() or not (p.get("revision") or "").strip(): ok = False; pistas.append("¿Quién mira el tablero y cada cuánto?")
    return {"ok": ok and cubiertos == len(INCIDENTES), "pistas": pistas}

LABORATORIO = {1: simular_contrato, 2: simular_registro, 3: simular_humo, 4: simular_latencia, 5: simular_drift, 6: simular_fuga, 7: simular_tablero}

# ═══════ REVISIÓN COMÚN DE LA FICHA DE EVIDENCIA: devuelve preguntas, no respuestas
ETAPAS = ("notebook", "paquete", "api", "registro", "dashboard")
def revisar_ficha(f):
    pistas = []
    for k, q in [("decision", "¿Qué decisión habilita?"), ("poblacion", "¿Cuál es la población?"), ("periodo", "¿Cuál es el período?"), ("granularidad", "¿Qué representa cada fila?"), ("objetivo", "¿Cuál es la variable objetivo?"), ("validacion", "¿Qué validaron y cómo evitaron fuga o sesgo?"), ("limite", "¿Qué no pueden afirmar?"), ("responsable_humano", "¿Qué persona responde por el servicio?"), ("versionado", "¿Cómo se versionan modelo y contrato?"), ("prueba_de_humo", "¿Cuál es la prueba de humo antes de dejar pasar tráfico?")]:
        if not (f.get(k) or "").strip(): pistas.append(q)
    ruta = f.get("ruta") or {}
    for e in ETAPAS:
        r = ruta.get(e) or {}
        if not (r.get("que_pasa") and r.get("control")): pistas.append(f"En la etapa «{e}» de la ruta, ¿qué pasa y qué control tiene?")
    rb = f.get("rollback") or {}
    if not (rb.get("disparador") and rb.get("responsable") and len(rb.get("pasos") or []) >= 3): pistas.append("¿Qué dispara la marcha atrás, quién decide y cuáles son sus pasos (tres o más)?")
    lat = f.get("latencia") or {}
    if not (lat.get("objetivo_p95_ms") and lat.get("como_se_mide")): pistas.append("¿Cuál es el objetivo de latencia y cómo se mide?")
    dr = f.get("drift") or {}
    if not (dr.get("senal") and dr.get("umbral") and dr.get("accion")): pistas.append("¿Qué señal de drift vigilan, con qué umbral y qué hacen cuando suena?")
    if len(f.get("cinco_pruebas") or []) < 5: pistas.append("¿Cuáles son las cinco pruebas? (Faltan pruebas)")
    if len(f.get("tablero_minimo") or []) < 3: pistas.append("¿Qué paneles tiene el tablero mínimo? (Tres o más)")
    if len(f.get("contrato_revisado") or []) < 3: pistas.append("¿Qué tres cambios o más le hicieron al contrato?")
    alt = f.get("alternativa_descartada") or {}
    if not (alt.get("cual") and alt.get("por_que")): pistas.append("¿Qué alternativa descartaron y por qué?")
    print(f"Revisión de la ficha: {len(pistas)} pregunta(s) pendiente(s).")
    for q in pistas: print("  👉", q)
    if not pistas: print("✔ Sin preguntas pendientes: la ficha se puede defender.")
    return pistas


CASOS_LISTOS = True

## Tu caso

Cambiá el número por el de tu equipo (1 a 7) y corré la celda: te muestra tu situación, lo que el caso declara y lo que no declara, y el cuidado a tener.

In [ ]:
# Los siete casos, las plantillas y las funciones para verlos (corré esta celda una sola vez)
# ═══════════════════════════ LOS SIETE DESAFÍOS · clase 4 · De modelo experimental a producción (NovaRetail)
ENTREGABLE = ["Diagrama de despliegue: notebook → paquete → API → registro → dashboard", "Contrato revisado, con sus cambios", "Cinco pruebas", "Tablero mínimo de monitoreo", "Decisión de rollback: disparador, responsable y pasos", "Una alternativa descartada por decisión"]
CASOS = {
 1: dict(titulo="El contrato de entrada", situacion="El CRM va a llamar al servicio con los datos de cada cliente. Hoy el contrato solo exige un id: cualquier otro dato puede faltar, venir mal o sobrar.",
         declarado="Hay un contrato borrador y 14 pedidos de prueba, entre válidos e inválidos.", no_declarado="Qué hace el modelo con un dato que falta o que viene mal, sin que nadie se entere.",
         foco="Campos obligatorios, rangos, valores permitidos y qué se rechaza.", cuidado="Aceptar campos extra o faltantes en silencio.", debrief="¿Qué evidencia necesitaría una persona ajena al equipo para confiar en que el contrato rechaza lo que debe?"),
 2: dict(titulo="Versionado y registro", situacion="Un cliente reclama una decisión que el servicio tomó hace 20 días y Gobierno pide reproducirla. Hoy el servicio solo guarda la probabilidad.",
         declarado="Hay dos versiones del modelo y un contrato versionado.", no_declarado="Qué hay que guardar de cada modelo y de cada predicción, por cuánto tiempo y para quién.",
         foco="La ficha de cada versión y el registro de cada predicción.", cuidado="No versionar el modelo.", debrief="¿Qué parte del registro depende de una política de privacidad todavía no confirmada?"),
 3: dict(titulo="Prueba de humo y rollback", situacion="Mañana sube una versión nueva del servicio. Una versión con un error de código ya llegó una vez a producción y nadie lo notó durante horas.",
         declarado="Hay una versión en producción y otra candidata; la prueba de humo corre antes de dejar pasar tráfico.", no_declarado="Con qué casos se prueba, qué diferencia es normal entre dos modelos y quién decide volver atrás.",
         foco="Los casos de la prueba, la tolerancia y la decisión de rollback.", cuidado="Probar solo con clientes típicos.", debrief="¿Qué control podría generar falsos positivos y cómo se administraría?"),
 4: dict(titulo="Latencia bajo carga", situacion="El CRM llama al servicio durante la atención de cada ticket y, de noche, otro proceso puntúa a todos los clientes.",
         declarado="El CRM genera unos 40 pedidos por segundo. El proceso nocturno puntúa 8.000 clientes y tiene 5 minutos.", no_declarado="Qué pasa en un pico de tickets y cuánto tarda una llamada según el tamaño del pedido.",
         foco="Individual o por lotes, tamaño del lote, timeout e instancias.", cuidado="Usar el mismo modo para el CRM y para el proceso nocturno.", debrief="¿Qué decisión cambiaría si la latencia requerida se redujera diez veces?"),
 5: dict(titulo="Drift y monitoreo", situacion="Con los meses cambian los clientes: aparecen más dormidos y un canal nuevo. El abandono real se conoce 90 días después.",
         declarado="Hay datos de los pedidos todos los días; el resultado real llega a los 90 días.", no_declarado="Qué cambio es normal y cuál no, y qué se hace cuando suena una alerta.",
         foco="Señales de entrada, de datos y de modelo, con su umbral y su acción.", cuidado="Esperar a los resultados reales para enterarse.", debrief="¿Qué componente concentra el mayor radio de impacto ante una falla?"),
 6: dict(titulo="Fuga y consistencia", situacion="El modelo entrena con todas las columnas de la tabla de clientes. En la prueba anda casi perfecto.",
         declarado="Hay catorce columnas candidatas para entrenar y el servicio recibe doce campos.", no_declarado="Cuáles de esas columnas existen en el momento de puntuar a un cliente.",
         foco="Qué columnas usar y cómo validar con clientes más nuevos.", cuidado="Confiar en una prueba que parece perfecta.", debrief="¿Qué otra columna podría estar contando la respuesta sin que nadie se dé cuenta?"),
 7: dict(titulo="El tablero mínimo", situacion="El servicio ya está en producción y Operaciones pide un tablero. Alguien propone sumar todo lo que se pueda medir.",
         declarado="Hay un catálogo de once indicadores posibles y siete incidentes que ya pasaron.", no_declarado="Cuáles de esos indicadores avisan de cada incidente y quién va a mirar el tablero.",
         foco="Qué paneles, quién los mira y cada cuánto.", cuidado="Medir lo fácil en lugar de lo que avisa.", debrief="¿Qué incidente de la lista no vería nadie si el tablero tuviera un panel menos?"),
}
PLANTILLAS = {
 1: '''parametros = {
    "campos_obligatorios": ["id_cliente"],         # ¿cuáles hacen falta para que el modelo puntúe?
    "rechazar_campos_extra": False,                # ¿qué se hace con un campo que el contrato no conoce?
    "rangos": {},                                  # por ejemplo {"edad": (18, 100)}: el mínimo y el máximo de cada campo numérico
    "valores_permitidos": {},                      # por ejemplo {"segmento": ["Esencial", "Plus", "Premium"]}
    "admite_nulo": [],                             # campos que pueden venir nulos
}''',
 2: '''parametros = {
    "por_modelo": [],        # qué se guarda de cada versión. Opciones: ''' + "version, algoritmo, auc_prueba, columnas_entrada, version_contrato, semilla, hash_datos_entrenamiento, filas_entrenamiento, fecha_entrenamiento, aprobado_por, commit_codigo, entorno" + '''
    "por_prediccion": [],    # qué se guarda de cada pedido. Opciones: ''' + "timestamp, request_id, version_modelo, version_contrato, hash_entrada, entrada_guardada, probabilidad, status, latencia_ms" + '''
    "retencion_dias": None,  # por cuánto tiempo se guarda la entrada, si la guardan
    "acceso": "",            # quién puede leerla
}''',
 3: '''parametros = {
    "casos": "tipicos",                # "tipicos", "borde" o "mezcla": con qué clientes se hace la prueba de humo
    "tolerancia": 0.05,                # diferencia máxima aceptada frente a las respuestas de la versión en producción
    "accion_si_falla": "",             # "rollback_automatico" o "alerta"
    "responsable": "",                 # quién decide, si la acción es una alerta
}''',
 4: '''parametros = {
    "interactivo": {"modo": "individual", "tamano_lote": 1, "instancias_base": 1, "autoescalado_hasta": 1},   # el CRM: "individual" o "lote"
    "masivo": {"tamano_lote": 1, "timeout_ms": 1000},                                                          # el proceso nocturno
    "objetivo_p95_ms": 300,
}''',
 5: '''parametros = [                          # una señal por línea: "psi_dias", "psi_score", "pct_422" o "auc_etiquetas"
    {"senal": "pct_422", "umbral": 0.05, "accion": "", "responsable": ""},
    # {"senal": "...", "umbral": ..., "accion": "...", "responsable": "..."},
]
# psi_* y pct_422 suben cuando algo cambia (más alto = más cambio); auc_etiquetas avisa si el AUC cae por debajo del umbral''',
 6: '''parametros = {
    "columnas": [],          # con qué columnas entrena el modelo. Opciones: ''' + "canal, segmento, region, edad, antiguedad_meses, compras_12m, ticket_promedio, dias_desde_ultima_compra, contactos_soporte_90d, usa_app, descuento_promedio, satisfaccion, satisfaccion_faltante, motivo_baja" + '''
}''',
 7: '''parametros = {
    "paneles": [],           # elegí del catálogo (no más de seis): ''' + "volumen_total, volumen_por_region, tasa_errores_422, latencia_p95, distribucion_scores, psi_variables, version_activa, auc_etiquetas, uso_de_cpu, usuarios_conectados, filas_en_tabla_de_clientes" + '''
    "responsable": "",       # quién mira el tablero
    "revision": "",          # cada cuánto
}''',
}
PLANTILLA_FICHA = '''ficha = {
    "decision": "",            # qué decisión habilita el servicio
    "poblacion": "", "periodo": "", "granularidad": "", "objetivo": "",     # población, período, qué representa cada fila y variable objetivo
    "ruta": {                  # notebook → paquete → API → registro → dashboard: qué pasa y qué control tiene cada etapa
        "notebook":  {"que_pasa": "", "control": ""},
        "paquete":   {"que_pasa": "", "control": ""},
        "api":       {"que_pasa": "", "control": ""},
        "registro":  {"que_pasa": "", "control": ""},
        "dashboard": {"que_pasa": "", "control": ""},
    },
    "versionado": "",          # cómo se versionan modelo y contrato
    "prueba_de_humo": "",      # qué se prueba antes de dejar pasar tráfico
    "rollback": {"disparador": "", "responsable": "", "pasos": ["", "", ""]},
    "latencia": {"objetivo_p95_ms": None, "como_se_mide": ""},
    "drift": {"senal": "", "umbral": None, "accion": ""},
    "responsable_humano": "",  # qué persona responde por el servicio
    "validacion": "",          # qué validaron y cómo evitaron fuga o sesgo
    "limite": "",              # qué no pueden afirmar
    "cinco_pruebas": ["", "", "", "", ""],
    "tablero_minimo": ["", "", ""],
    "contrato_revisado": ["", "", ""],      # los cambios que le hicieron al contrato
    "alternativa_descartada": {"cual": "", "por_que": ""},
}'''
def ver_caso():
    c = CASOS[MI_EQUIPO]; print(f"EQUIPO {MI_EQUIPO} · {c['titulo']}\n")
    for etiqueta, clave in [("SITUACIÓN", "situacion"), ("EL CASO DECLARA", "declarado"), ("EL CASO NO DECLARA", "no_declarado"), ("FOCO", "foco"), ("CUIDADO CON", "cuidado"), ("PARA EL DEBRIEF", "debrief")]: print(f"{etiqueta}\n  {c[clave]}\n")
    print("QUÉ ENTREGAN")
    for e in ENTREGABLE: print("  •", e)
def ver_plantilla(): print(PLANTILLAS[MI_EQUIPO])
def ver_plantilla_ficha(): print(PLANTILLA_FICHA)
def simular(parametros):
    print(f"━━ Laboratorio del equipo {MI_EQUIPO}: {CASOS[MI_EQUIPO]['titulo']}\n"); res = LABORATORIO[MI_EQUIPO](parametros)
    print("\n" + ("✔ La simulación no devuelve preguntas." if res["ok"] and not res["pistas"] else "Preguntas que devuelve la simulación:"))
    for q in res["pistas"]: print("  👉", q)
    return res

In [ ]:
MI_EQUIPO = 1   # 👉 cambiá este número por el de tu equipo (1 a 7)
ver_caso()

## Las herramientas del kit

| Función | Qué hace |
| --- | --- |
| `ver_caso()` | Muestra tu situación y lo que tenés que entregar |
| `ver_plantilla()` | Muestra la plantilla de parámetros de **tu laboratorio** |
| `simular(parametros)` | Corre la simulación de tu caso con tus decisiones y **devuelve preguntas**, no respuestas |
| `ver_plantilla_ficha()` | Muestra la plantilla de la ficha de evidencia |
| `revisar_ficha(ficha)` | Revisa tu ficha y **devuelve preguntas**, no respuestas |

Los resultados salen de datos generados por código y de reglas simples. **Las latencias del caso 4 son estimaciones del kit, no mediciones de un servicio real.**

## Tu turno

### Paso 1 · La decisión y la población

Escriban, en esta celda de texto, **qué decisión habilita** el servicio, **cuál es la población y el período**, **qué representa cada fila** y **cuál es la variable objetivo**.

- **Decisión:**
- **Población y período:**
- **Granularidad y variable objetivo:**
- **Supuestos que declaramos:**

### Paso 2 · El laboratorio de tu caso

Cada caso tiene una simulación que **prueba una decisión de ustedes** contra una situación real. Corran `ver_plantilla()`, copien la plantilla en una celda nueva, completen **sus** valores y corran `simular(parametros)`.

No hay una respuesta única: cada simulación devuelve **preguntas** sobre lo que su diseño no resuelve. Cambien un valor y vuelvan a correrla.

In [ ]:
# 👉 TU TURNO · Paso 2: el laboratorio de tu caso
ver_plantilla()

# parametros = ...   (copiá la plantilla de arriba y completala)
# resultado = simular(parametros)

### Paso 3 · ¿Qué decisión cambió el resultado?

Miren lo que devolvió la simulación y respondan, en una frase cada una:

- **¿Qué decisión de ustedes cambió el resultado?**
- **¿Qué costo aceptaron para conseguirlo?**
- **¿Qué alternativa descartaron y por qué?**

### Paso 4 · La ficha de evidencia

Ahora armen la ficha completa de su ruta. Corran `ver_plantilla_ficha()`, copien la plantilla y completenla con lo que decidieron en el laboratorio.

In [ ]:
# 👉 TU TURNO · Paso 4: la ficha de evidencia
ver_plantilla_ficha()

# ficha = ...   (copiá la plantilla de arriba y completala)

### Paso 5 · La revisión final

`revisar_ficha` **no corrige**: devuelve preguntas. Respóndanlas y vuelvan a correrla hasta que no queden pendientes.

In [ ]:
# 👉 TU TURNO · Paso 5: revisar la ficha
# pistas = revisar_ficha(ficha)

## Evidencia de entrega

**Equipo: ________ · Desafío: ________**

Respondan, en una frase cada una:

1. **¿Qué decisión habilita el servicio?**
2. **¿Cuál es la población y el período?**
3. **¿Cómo evitaron fuga o sesgo?**
4. **¿Qué validaron y contra qué referencia?**
5. **¿Qué no pueden afirmar?**
6. **¿Qué decidieron descartar y por qué?**

### Cómo se evalúa

Se considera logrado cuando el equipo responde la pregunta y la decisión definidas, declara población, período, granularidad y variable objetivo, conserva una ruta reproducible, compara contra una referencia con métricas acordes al costo del error, incluye controles de calidad, fuga, sesgo y generalización, separa evidencia, interpretación y recomendación, y documenta una limitación y un siguiente paso.

| Nivel | Qué se ve |
| --- | --- |
| **Insuficiente** | Empaqueta el notebook tal cual; no dice qué valida ni qué recupera |
| **Inicial** | Técnicamente posible, pero acepta campos extra en silencio, no versiona o no dice cómo vuelve atrás |
| **Sólido** | Contrato que rechaza, versionado, prueba de humo con casos de borde y un tablero con acción y responsable |
| **Defendible** | Todo lo anterior, más una alternativa descartada con su costo, una falla simulada y una defensa clara |

**La devolución escrita** incluye una fortaleza, una brecha prioritaria y una mejora verificable.